This code is for making a list all valid dates where there is an image with less than 50% cloud coverage for every MGRS tile or Path Row combination, and making random points for sampling. These lists of valid dates make filtering for instances where there are overlapping images much quicker.

In [17]:
import os,re,ast,time,random,rasterio,ee,glob,warnings
from datetime import datetime as _dt
from itertools import product
import concurrent.futures as fut
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from shapely.geometry import Point
from shapely.strtree import STRtree
from shapely.ops import unary_union
from shapely.prepared import prep
from rasterio.features import geometry_mask
from joblib import Parallel, delayed
from tqdm.auto import tqdm

ee_project="ee-gamr05055"
L7COL='LANDSAT/LE07/C02/T1_L2'
L8COL="LANDSAT/LC08/C02/T1_L2"
S30COL="NASA/HLS/HLSS30/v002"
ee.Authenticate()
ee.Initialize(project=ee_project)

### Step 1. Making all_mgrs and all_pathrows
This code makes a textfile containing of all MRGS and path/row combinations, which will be sorted through later.

In [18]:
# Points are alreadly located at L7L8 intersections
def list_tile_intersections(points,wrs,mgrs,path_col='PATH',row_col='ROW',mgrs_col=None,keep_point_cols=True,n_jobs=-1):
    _g=lambda x:gpd.read_file(x) if isinstance(x,(str,os.PathLike)) else x
    pts=_g(points).copy(); wrs=_g(wrs).copy(); mgrs=_g(mgrs).copy()
    if mgrs_col is None:
        for c in('MGRS','mgrs','NAME','Name','name','TILE','Tile','ID','grid','GRID','MGRS_TILE','Name'):
            if c in mgrs.columns: mgrs_col=c; break
        else: raise KeyError('mgrs_col not found')
    wrs=wrs[[path_col,row_col,'geometry']].to_crs(pts.crs); mgrs=mgrs[[mgrs_col,'geometry']].to_crs(pts.crs)
    siw,sim=wrs.sindex,mgrs.sindex
    uniq=lambda a:sorted(pd.Series(a).dropna().unique().tolist())
    def f(i,p):
        cw=list(siw.query(p)) if hasattr(siw,'query') else list(siw.intersection(p.bounds))
        cm=list(sim.query(p)) if hasattr(sim,'query') else list(sim.intersection(p.bounds))
        w=wrs.iloc[cw]; m=mgrs.iloc[cm]
        if len(w): w=w.loc[w.geometry.intersects(p)]
        if len(m): m=m.loc[m.geometry.intersects(p)]
        return i, (uniq(w[path_col]) if len(w) else []), (uniq(w[row_col]) if len(w) else []), (uniq(m[mgrs_col]) if len(m) else [])
    res=Parallel(n_jobs=n_jobs,prefer='threads',batch_size=256)(delayed(f)(i,p) for i,p in zip(pts.index,pts.geometry))
    out=(pts if keep_point_cols else pts[['geometry']]).copy(); n=len(out)
    out['paths']=[[] for _ in range(n)]; out['rows']=[[] for _ in range(n)]; out['mgrs']=[[] for _ in range(n)]
    for i,pa,ro,mg in res: out.at[i,'paths']=pa; out.at[i,'rows']=ro; out.at[i,'mgrs']=mg
    return gpd.GeoDataFrame(out,geometry='geometry',crs=pts.crs)

In [19]:
def build_pathrow_and_mgrs_lists(gdf, path_col='paths', row_col='rows', mgrs_col='mgrs'):
    pr=set(); mg=set()
    for paths,rows,tiles in gdf[[path_col,row_col,mgrs_col]].itertuples(index=False):
        if isinstance(paths,(list,tuple)) and isinstance(rows,(list,tuple)) and paths and rows:
            p={int(p) for p in paths if pd.notna(p)}
            r={int(x) for x in rows  if pd.notna(x)}
            pr.update(product(p,r))
        if isinstance(tiles,(list,tuple)) and tiles:
            mg.update({str(t).strip().upper() for t in tiles if pd.notna(t) and str(t).strip()})
    pathrowlist=sorted(pr,key=lambda t:(t[0],t[1]))
    mgrslist=sorted(mg)
    return pathrowlist, mgrslist

def write_pathrow_txt(pathrowlist, out_path, sep=','):
    with open(out_path,'w') as f:
        f.writelines(f"{p}{sep}{r}\n" for p,r in pathrowlist)
def write_mgrs_txt(mgrslist, out_path):
    with open(out_path,'w') as f:
        f.writelines(f"{tile}\n" for tile in mgrslist)

In [13]:
point_gdf=list_tile_intersections(
    r'E:\GIS\Landcover_sampling\Reference\All_points_MGRS_WRS.shp',
    r"E:\GIS\Landsat Normalization\Landsat_WRS_index\WRS2_descending.shp",
    r"E:\GIS\Landsat Normalization\Landsat_WRS_index\sentinel_2_index_shapefile.shp",
    path_col='PATH',row_col='ROW',mgrs_col='Name',keep_point_cols=True,n_jobs=-1)
pathrowlist,mgrslist=build_pathrow_and_mgrs_lists(point_gdf)
write_mgrs_txt(mgrslist, r"E:\GIS\Landcover_sampling\Reference\All_mgrs.txt")
write_pathrow_txt(pathrowlist, r"E:\GIS\Landcover_sampling\Reference\All_pathrows.txt", sep=',')

'point_gdf=list_tile_intersections(\n    r\'E:\\GIS\\Landcover_sampling\\Reference\\All_points_MGRS_WRS.shp\',\n    r"E:\\GIS\\Landsat Normalization\\Landsat_WRS_index\\WRS2_descending.shp",\n    r"E:\\GIS\\Landsat Normalization\\Landsat_WRS_index\\sentinel_2_index_shapefile.shp",\n    path_col=\'PATH\',row_col=\'ROW\',mgrs_col=\'Name\',keep_point_cols=True,n_jobs=-1)\n#pathrowlist,mgrslist=build_pathrow_and_mgrs_lists(point_gdf)\n#write_mgrs_txt(mgrslist, r"E:\\GIS\\Landcover_sampling\\Reference\\All_mgrs.txt")\n#write_pathrow_txt(pathrowlist, r"E:\\GIS\\Landcover_sampling\\Reference\\All_pathrows.txt", sep=\',\')'

### Step 2.1 Make L7_dates, L8_dates, S30_dates
The below code makes csv files containing path/row combinations or MGRS tiles, and a list of all dates that have valid images. 

In [21]:
def read_pathrow_txt(txt_path, sep=','):
    with open(txt_path) as f:
        return [tuple(map(int, line.strip().split(sep))) for line in f if line.strip()]
def read_mgrs_txt(txt_path):
    with open(txt_path,"r") as f:
        return [line.strip().upper() for line in f if line.strip()]
def _logfile(col):
    s=str(col).upper()
    return ("L7logs.txt","L7") if "LE07" in s else (("L8logs.txt","L8") if ("LC08" in s or "LC09" in s) else (("S30 logs.txt","S30") if "HLSS30" in s else ("IC_logs.txt","IC")))
def _log(dir,f,txt):
    os.makedirs(dir,exist_ok=True); open(os.path.join(dir,f),"a",encoding="utf-8").write(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {txt}\n")
class _NC: 
    def __enter__(self): return None
    def __exit__(self,*a): return False
def _tqdm_joblib(t):
    if not t: return _NC()
    class CB(joblib.parallel.BatchCompletionCallBack):
        def __call__(self,*a,**k): 
            try: t.update(n=self.batch_size)
            finally: return super().__call__(*a,**k)
    old=joblib.parallel.BatchCompletionCallBack; joblib.parallel.BatchCompletionCallBack=CB
    class CM:
        def __enter__(self): return t
        def __exit__(self,*a): joblib.parallel.BatchCompletionCallBack=old; t.close()
    return CM()
def landsat_dates_for_pr(inp,log_dir,col,start="2016-01-01",end="2022-01-01",cloud_lte=50,n_jobs=-1,
                         retries=3,backoff_s=1.5):
    logname,tag=_logfile(col); t0=time.time()
    try: ee.Initialize(project=ee_project) if ee_project else ee.Initialize()
    except Exception: ee.Authenticate(); ee.Initialize(project=ee_project) if ee_project else ee.Initialize()
    if hasattr(inp,"columns"):
        d={c.lower():c for c in inp.columns}; P=d.get("wrs_path",d.get("path")); R=d.get("wrs_row",d.get("row"))
        prs=[(int(p),int(r)) for p,r in inp[[P,R]].to_numpy()]
    else:
        a=np.asarray(inp,object)
        if a.ndim==2 and a.shape[1]==2: prs=[(int(p),int(r)) for p,r in a.tolist()]
        elif a.ndim==2 and a.shape[0]==2: prs=[(int(p),int(r)) for p,r in a.T.tolist()]
        else: prs=[(int(p),int(r)) for p,r in a.tolist()]
    uniq=list(dict.fromkeys(prs))
    _log(log_dir,logname,f"{tag}: start col={col} range=[{start}..{end}] cloud_lte={cloud_lte} n_jobs={n_jobs}")
    _log(log_dir,logname,f"{tag}: input={len(prs)} unique_keys={len(uniq)}")
    def _fetch(p,r):
        for k in range(retries):
            try:
                ic=(ee.ImageCollection(col)
                    .filter(ee.Filter.eq("WRS_PATH",int(p)))
                    .filter(ee.Filter.eq("WRS_ROW",int(r)))
                    .filterDate(start,end)
                    .filter(ee.Filter.lte("CLOUD_COVER",cloud_lte)))
                d=ic.aggregate_array("DATE_ACQUIRED").getInfo()
                if not d:
                    ts=ic.aggregate_array("system:time_start").getInfo()
                    d=[ee.Date(t).format("YYYY-MM-dd").getInfo() for t in ts] if ts else []
                return sorted(set(d))
            except Exception as e:
                if any(z in str(e).lower() for z in ("429","quota","rate")): time.sleep(backoff_s*(1.5**k)+random.random()); continue
                raise
        return []
    with _tqdm_joblib(tqdm(total=len(uniq),desc=f"{tag} dates",unit="key")):
        vals=Parallel(n_jobs=n_jobs,prefer="threads")(delayed(_fetch)(p,r) for p,r in uniq)
    lut=dict(zip(uniq,vals))
    out=pd.DataFrame(prs,columns=["WRS_PATH","WRS_ROW"])
    out["dates_by_pr"]=[lut[(int(p),int(r))] for p,r in out[["WRS_PATH","WRS_ROW"]].to_numpy()]
    out=out[out["dates_by_pr"].map(bool)].reset_index(drop=True)
    _log(log_dir,logname,f"{tag}: output_rows={len(out)} elapsed={time.time()-t0:.2f}s")
    return out

In [32]:
pathrowlist = read_pathrow_txt(r"E:\GIS\Landcover_sampling\Reference\All_pathrows.txt")
L7_dates=landsat_dates_for_pr(pathrowlist,log_dir=r"E:\GIS\Landcover_sampling\Reference", L7COL)#Get L7 dates
L7_dates.to_csv(r'E:\GIS\Landcover_sampling\Reference\L7_dates50.csv', index=False)
L8_dates=landsat_dates_for_pr(pathrowlist,log_dir=r"E:\GIS\Landcover_sampling\Reference",L8COL) #Get L8 dates
L8_dates.to_csv(r'E:\GIS\Landcover_sampling\Reference\L8_dates50.csv', index=False)

L7 dates:   0%|          | 0/9985 [00:00<?, ?key/s]

L8 dates:   0%|          | 0/9985 [00:00<?, ?key/s]

KeyboardInterrupt: 

## Step 2.2 Sen2 Dates
Since Sentinel-2 has a lot more tiles and dates to sort through, we have a slightly different approach. This code takes a lot longer than the Landsat filtering. It returns all the valid dates of images for all of the MGRS tiles.

In [30]:
def mgrs_dates_for_tiles(mgrslist,temp_csv,log_dir,final_csv,col="NASA/HLS/HLSS30/v002",start="2016-01-01",end="2022-01-01",
                         cloud_lte=50,n_jobs=-1,retries=3,backoff_s=1.5,
                         existing_csv=None,chunk=10,timeout_s=120,ee_project=None):
    logname,tag=_logfile(col); t0=time.time()
    try: ee.Initialize(project=ee_project) if ee_project else ee.Initialize()
    except Exception: ee.Authenticate(); ee.Initialize(project=ee_project) if ee_project else ee.Initialize()
    san=lambda s:(str(s).strip().upper()[1:] if str(s).strip().upper().startswith("T") else str(s).strip().upper())
    if hasattr(mgrslist,"columns"):
        cn=[c.lower() for c in mgrslist.columns]; c="mgrs" if "mgrs" in cn else mgrslist.columns[0]; tiles=[san(v) for v in mgrslist[c].tolist()]
    else: tiles=[san(v) for v in np.asarray(mgrslist,object).ravel().tolist()]
    uniq=list(dict.fromkeys(tiles))
    _log(log_dir,logname,f"{tag}: start col={col} range=[{start}..{end}] cloud_lte={cloud_lte} n_jobs={n_jobs}")
    _log(log_dir,logname,f"{tag}: input={len(tiles)} unique_keys={len(uniq)}")
    parse=lambda x:(x if isinstance(x,list) else ([] if (x is None or (isinstance(x,float) and np.isnan(x)) or x=="") else (list(ast.literal_eval(x)) if isinstance(x,str) else [])))
    src_csv=existing_csv if existing_csv else (temp_csv if os.path.isfile(temp_csv) else None)
    done_lut={}
    if src_csv and os.path.isfile(src_csv):
        try:
            df0=pd.read_csv(src_csv)
            if "mgrs" in df0.columns and "dates_by_mgrs" in df0.columns:
                done_lut={san(m):parse(d) for m,d in zip(df0["mgrs"],df0["dates_by_mgrs"])}
        except Exception: pass
    to_process=[t for t in uniq if t not in done_lut]
    _log(log_dir,logname,f"{tag}: skip_existing={len(uniq)-len(to_process)}")
    def _fast_dates(t):
        ic=(ee.ImageCollection(col)
            .filter(ee.Filter.eq("MGRS_TILE_ID",t))
            .filterDate(start,end)
            .filter(ee.Filter.lte("CLOUD_COVERAGE",cloud_lte)))
        try:
            d=ic.aggregate_array("DATE_ACQUIRED").getInfo()
            if not d:
                ts=ic.aggregate_array("system:time_start").getInfo()
                d=[ee.Date(tt).format("YYYY-MM-dd").getInfo() for tt in ts] if ts else []
            return sorted(set(d))
        except Exception: return None
    def _fast_dates_with_timeout(t):
        with fut.ThreadPoolExecutor(max_workers=1) as ex:
            f=ex.submit(_fast_dates,t)
            try: return f.result(timeout=timeout_s)
            except fut.TimeoutError: return None
            except Exception: return None
    def _slow_yearwise(t):
        y0,y1=int(start[:4]),int(end[:4]); out=[]
        for y in range(y0,y1+1):
            s,f=f"{y}-01-01",(f"{y+1}-01-01" if y<y1 else end)
            ic=(ee.ImageCollection(col)
                .filter(ee.Filter.eq("MGRS_TILE_ID",t))
                .filterDate(s,f)
                .filter(ee.Filter.lte("CLOUD_COVERAGE",cloud_lte)))
            try: d=ic.aggregate_array("DATE_ACQUIRED").getInfo() or []
            except Exception: d=[]
            if not d:
                try:
                    ts=ic.aggregate_array("system:time_start").getInfo() or []
                    d=[ee.Date(tt).format("YYYY-MM-dd").getInfo() for tt in ts] if ts else []
                except Exception: d=[]
            if d: out.extend(d)
        return sorted(set(out))
    def _fetch(t):
        for k in range(retries):
            try:
                d=_fast_dates_with_timeout(t)
                if d is None: raise RuntimeError("fast timeout/err")
                return d if d else _slow_yearwise(t)
            except Exception as e:
                if any(z in str(e).lower() for z in ("429","quota","rate")):
                    time.sleep(backoff_s*(1.5**k)+random.random()); continue
                try: return _slow_yearwise(t)
                except Exception: time.sleep(1+0.5*k)
        return []
    new_lut={}; pbar=tqdm(total=len(to_process),desc=f"{tag} dates",unit="tile")
    os.makedirs(os.path.dirname(temp_csv),exist_ok=True)
    for i in range(0,len(to_process),max(1,chunk)):
        batch=to_process[i:i+chunk]
        vals=Parallel(n_jobs=n_jobs,prefer="threads")(delayed(_fetch)(t) for t in batch)
        for t,v in zip(batch,vals): new_lut[t]=v
        bdf=pd.DataFrame({"mgrs":batch,"dates_by_mgrs":[new_lut[t] for t in batch]})
        bdf=bdf[bdf["dates_by_mgrs"].map(bool)]
        if not bdf.empty: bdf.to_csv(temp_csv,mode="a",header=not os.path.isfile(temp_csv),index=False)
        pbar.update(len(batch))
    pbar.close()
    lut={**done_lut,**new_lut}
    out=pd.DataFrame({"mgrs":tiles,"dates_by_mgrs":[lut.get(t,[]) for t in tiles]})
    out=out[out["dates_by_mgrs"].map(bool)].reset_index(drop=True)
    os.makedirs(os.path.dirname(final_csv),exist_ok=True); out.to_csv(final_csv,index=False)
    _log(log_dir,logname,f"{tag}: output_rows={len(out)} elapsed={time.time()-t0:.2f}s wrote={final_csv} temp={temp_csv}")
    return out

In [31]:
mgrslist = read_mgrs_txt(r"E:\GIS\Landcover_sampling\Reference\All_mgrs.txt") #list of MGRS tiles are located
S30_dates=mgrs_dates_for_tiles(mgrslist,r"E:\GIS\Landcover_sampling\Reference\S30_datestemp50.csv"
                              r"E:\GIS\Landcover_sampling\Reference",
                               r"E:\GIS\Landcover_sampling\Reference\S30_dates50.csv") #we also create a temp df incase progress is interrupted

S30 dates:   0%|          | 0/18050 [00:00<?, ?tile/s]

KeyboardInterrupt: 

### Step 3. Creating points using code from LEOHS
This code make points located in L7 and L8 overlaps using a similar approach to LEOHS, just slightly optimized to make the processing more efficient. These points are create on an equal area projected -- if they are within an east/west WRS overlap then they stay, otherwised they are moved to the nearest east/west WRS overlap and randomly placed within it.

- WRS_world_overlaps_c.shp is all of the east/west overlaps clipped to the extent of the world. If your study area is smaller you can clip this to the extent.
- World_Countries_Generalized.shp is the input study area in the below code.

In [ ]:
EQ, WGS = "EPSG:8857", "EPSG:4326"
def _fix_invalid(gdf):
    gdf=gdf.copy()
    try:
        from shapely.validation import make_valid; gdf["geometry"]=gdf.geometry.apply(make_valid)
    except Exception:
        gdf["geometry"]=gdf.buffer(0)
    return gdf[(~gdf.geometry.is_empty)&gdf.is_valid]
def generate_equalA_points(aoi, n):
    print("Generating equal-area random points...")
    aoi=aoi.to_crs(EQ); tf=rasterio.transform.from_bounds(*aoi.total_bounds,10000,10000)
    mask=geometry_mask([g for g in aoi.geometry], transform=tf, invert=True, out_shape=(10000,10000))
    r,c=np.where(mask)
    if len(r)<n: raise ValueError(f"AOI grid has {len(r)} candidate cells for {n} points; enlarge grid or lower n")
    idx=np.random.choice(len(r), size=n, replace=False); x,y=rasterio.transform.xy(tf, r[idx], c[idx])
    return gpd.GeoDataFrame(geometry=[Point(xx,yy) for xx,yy in zip(x,y)], crs=EQ)
def _rand_point_in(poly, rng, tries=10_000):
    minx,miny,maxx,maxy=poly.bounds
    for _ in range(tries):
        p=Point(rng.uniform(minx,maxx), rng.uniform(miny,maxy))
        if poly.contains(p): return p
    return poly.representative_point()
def _nearest_poly_index_for_each(outs, geoms):
    tree=STRtree(geoms); n=len(outs); chosen=np.full(n,-1,int)
    try:
        try: L,R=tree.query_nearest(outs, all_matches=False)
        except TypeError: L,R=tree.query_nearest(outs)
        for li,ri in zip(map(int,L),map(int,R)):
            if 0<=li<n and chosen[li]==-1: chosen[li]=ri
    except AttributeError:
        if hasattr(tree,"nearest_all"):
            L,R=tree.nearest_all(outs)
            for li,ri in zip(map(int,L),map(int,R)):
                if 0<=li<n and chosen[li]==-1: chosen[li]=ri
    if np.any(chosen==-1):
        gid={id(g):i for i,g in enumerate(geoms)}
        for i in np.where(chosen==-1)[0]:
            p=outs[i]; cand=list(tree.query(p))
            if not cand:
                for r in (1_000,10_000,100_000,1_000_000,10_000_000,40_000_000):
                    cand=list(tree.query(p.buffer(r))); 
                    if cand: break
            chosen[i]=gid[id(min(cand,key=lambda gg:p.distance(gg)))] if cand else 0
    return chosen
def process_points(aoi_pts, overlaps, seed=71):
    print("Processing points (checking overlap coverage)...")
    rng=np.random.default_rng(seed)
    overlaps=_fix_invalid(overlaps.to_crs(EQ))
    if overlaps.empty: raise ValueError("Overlap layer empty after CRS/validity fix.")
    aoi=aoi_pts.to_crs(EQ).reset_index(drop=True); ids=np.arange(len(aoi))
    keep=gpd.sjoin(aoi.assign(_id=ids), overlaps[["geometry"]], how="left", predicate="within")[["_id","index_right"]]
    ins_ids=np.unique(keep.loc[keep.index_right.notna(), "_id"].astype(int).to_numpy())
    out_ids=np.setdiff1d(ids, ins_ids, assume_unique=True)
    print(f"{ins_ids.size} inside, {out_ids.size} outside (unique ids).")
    inside=aoi.iloc[ins_ids][["geometry"]].copy()
    if out_ids.size:
        outs=list(aoi.geometry.iloc[out_ids].values); geoms=list(overlaps.geometry.values)
        chosen=_nearest_poly_index_for_each(outs, geoms)
        moved=[_rand_point_in(geoms[j], rng) for j in tqdm(chosen, total=len(chosen), desc="Moving outside points")]
        moved_gdf=gpd.GeoDataFrame(geometry=moved, crs=EQ)
        res=gpd.GeoDataFrame(pd.concat([inside, moved_gdf], ignore_index=True), geometry="geometry", crs=EQ)
    else:
        res=inside.set_crs(EQ)
    assert len(res)==len(aoi), f"Count drifted: got {len(res)}, expected {len(aoi)}"
    return res
def create_overlap_points_gdf(points_gdf, wrs_gdf):
    print("Creating overlap-only subset (QA)...")
    pts=points_gdf.to_crs(wrs_gdf.crs); wrs=_fix_invalid(wrs_gdf)
    j=gpd.sjoin(pts, wrs[["geometry"]], how="inner", predicate="within")[["geometry"]]
    #j = j.sample(frac=1, random_state=100).reset_index(drop=True)  # randomise
    return gpd.GeoDataFrame(j.drop_duplicates(subset="geometry").reset_index(drop=True), geometry="geometry", crs=wrs.crs)
def run_workflow(full_AOI, sample_points_n, frequency_gdf, seed=71):
    aoi_pts=generate_equalA_points(full_AOI, sample_points_n)
    sample_eq=process_points(aoi_pts, frequency_gdf, seed=seed)
    sample_wgs=sample_eq.to_crs(WGS)
    overlap_subset=create_overlap_points_gdf(sample_wgs, frequency_gdf)
    print(f"Workflow complete. Sample_points: {len(sample_wgs):,} (target {sample_points_n:,})")
    return sample_wgs, overlap_subset

In [ ]:
# this creates 1million points
Sample_points_gdf, overlap_points_gdf = run_workflow(
    gpd.read_file(r'E:\GIS\Landsat Normalization\Test_AOIS\World_Countries_Generalized.shp'),
    1_000_000,
    gpd.read_file(r'E:\GIS\Landsat Normalization\Landsat_WRS_index\WRS_world_overlaps_c.shp'))
print("Writing final shapefile...")
Sample_points_gdf.to_file(r'E:\GIS\Landcover_sampling\Initial Points\1mill.shp', driver='ESRI Shapefile')
print("Done.")

In [ ]:
### The input for the next step is a folder that contains a shapefile.
### There are no temp files for the next stage, so you might want to break up the point shapefile into chuncks.
### Place each chunk into its own folder
def split_20(in_shp):
    G=gpd.read_file(in_shp); base=os.path.splitext(os.path.basename(in_shp))[0]; d=os.path.dirname(in_shp)
    for i,ix in enumerate(np.array_split(range(len(G)),20),1):
        if len(ix): gpd.GeoDataFrame(G.iloc[ix],crs=G.crs).to_file(os.path.join(d,f"{base}_({i}).shp"))
split_20(r'E:\GIS\Landcover_sampling\Initial Points\2mill_init\1mil.shp')
### If you break up the shapefile into chuncks, you can stitch them back together with this.
def merge_folder(in_dir,out_shp):
    files=sorted(glob.glob(os.path.join(in_dir,"*.shp"))); dfs=[gpd.read_file(f) for f in files]
    g=gpd.GeoDataFrame(pd.concat(dfs,ignore_index=True),crs=(dfs[0].crs if dfs else None)); g.to_file(out_shp)
merge_folder(r'E:\GIS\Landcover_sampling\Intermediate\Tostitch',r'E:\GIS\Landcover_sampling\Intermediate\1mil.shp')